In [ ]:
%%capture
# Unsloth = fast QLoRA on the free T4. This pulls in transformers/peft/trl/bitsandbytes too.
!pip install unsloth
# Force the latest Unsloth (Colab's cached version is often stale)
!pip install --upgrade --no-cache-dir --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Logging + data
!pip install wandb        # experiment logging (we'll configure this later)

# If !pip install unsloth ever throws dependency errors on Colab, use this explicit version instead:
# %%capture
# !pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
# !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
# !pip install --no-deps unsloth wandb

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset, Dataset

import json
from pathlib import Path

# Mount Google Drive — checkpoint here so a Colab disconnect doesn't wipe progress
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)





In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
import json

# Phase-4 v2 fix: PROMPT AUGMENTATION.
# Rotate the system prompt per-example over {full schema, minimal, none} so the
# 8-section format + no-code-leak binds to the TASK, not to the schema sitting in the prompt.
SYSTEM_PROMPT = (
    "You are a DSA reasoning coach. Given a programming problem, teach the user how to "
    "DERIVE the solution. Respond using exactly these sections in order:
"
    "## Observations
## Brute force
## Bottleneck
## Key insight
"
    "## Pattern
## Optimized approach
## Complexity
## Generalizable lesson
"
    "Explain the thinking that leads to the approach. Do not write full code unless asked."
)
MINIMAL_PROMPT = "You are a helpful DSA tutor."

def system_variant(i):
    m = i % 3
    return SYSTEM_PROMPT if m == 0 else (MINIMAL_PROMPT if m == 1 else None)

def build_dataset(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    texts = []
    for i, row in enumerate(rows):
        sysmsg = system_variant(i)
        msg = ([{"role": "system", "content": sysmsg}] if sysmsg else []) + [
            {"role": "user",      "content": row["problem"]},
            {"role": "assistant", "content": row["derivation"]},
        ]
        texts.append(tokenizer.apply_chat_template(msg, tokenize=False))
    return Dataset.from_dict({"text": texts})

train_dataset = build_dataset("train.jsonl")
val_dataset   = build_dataset("val.jsonl")
print("train:", len(train_dataset), "val:", len(val_dataset))

# SANITY: [0]=full schema, [1]=minimal "helpful DSA tutor", [2]=NO system message
for k in range(3):
    print(f"
----- example {k} -----
", train_dataset[k]["text"][:300])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",     # the column name we created
        max_seq_length = max_seq_length, # 2048, from when we loaded the model

        # --- batch size ---
        per_device_train_batch_size = 1,   # examples per GPU step (T4 can't fit many)
        gradient_accumulation_steps = 8,   # accumulate 4 steps -> effective batch = 2*4 = 8

        # --- how long to train ---
        num_train_epochs = 3,              # passes over the 73 examples
        warmup_steps = 5,
        learning_rate = 2e-4,              # standard for LoRA
        lr_scheduler_type = "linear",
        weight_decay = 0.01,

        # --- precision (T4 = fp16) ---
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",              # memory-efficient optimizer

        # --- logging + eval + saving ---
        logging_steps = 1,                 # print loss every step
        eval_strategy = "steps",
        eval_steps = 5,                    # check val loss every 5 steps
        save_strategy = "steps",
        save_steps = 10,
        output_dir = "/content/drive/MyDrive/dsa-coach/outputs",  # checkpoints -> Drive
        report_to = "none",                # we'll switch this to W&B later

        seed = 42,
    ),
)

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 73 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
5,1.591004,1.580576
10,1.095439,1.426161
15,1.318912,1.310756
20,1.203658,1.235158
25,1.201050,1.194238
30,1.169822,1.179574


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dsa-coach/outputs/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dsa-coach/outputs/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dsa-coach/outputs/checkpoint-30/tokenizer_config.json.


In [ ]:

model.save_pretrained("/content/drive/MyDrive/dsa-coach/lora_7b_v2")
tokenizer.save_pretrained("/content/drive/MyDrive/dsa-coach/lora_7b_v2")
print("saved!")

## --- Eval cells below (Phase 3-style) ---
Run these in a **separate runtime** AFTER training, not during. They still reference 3B paths / no augmentation — update before reuse.

In [ ]:
%%capture
!pip install unsloth
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "/content/drive/MyDrive/dsa-coach/lora_7b_v2",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

FastLanguageModel.for_inference(model)
print("ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [ ]:
SYSTEM_PROMPT = (
    "You are a DSA reasoning coach. Given a programming problem, teach the user how to "
    "DERIVE the solution. Respond using exactly these sections in order:\n"
    "## Observations\n## Brute force\n## Bottleneck\n## Key insight\n"
    "## Pattern\n## Optimized approach\n## Complexity\n## Generalizable lesson\n"
    "Explain the thinking that leads to the approach. Do not write full code unless asked."
)

In [ ]:
messages=[
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":"You are given an array `nums` where each element represents the amount of money in a house. The houses are arranged in a circle, so the first and last houses are neighbors. You may rob any subset of houses, but you cannot rob two adjacent houses because the alarm will trigger. Return the maximum total amount of money you can collect without alerting the alarm."}
]
inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors="pt",return_dict=True).to("cuda")



In [ ]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True)

output = model.generate(
    input_ids=inputs,
    streamer=streamer,
    max_new_tokens=1024,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    use_cache=True,
)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


## Observations
- We need to pick a subset of non-consecutive houses. This is a classic “Knapsack with circular constraint” problem.
- If we were only dealing with a linear sequence (not circular), we would use a 2D DP table. However, since the houses form a circle, we must be careful about the first and last houses. The first house can be chosen or not chosen independently of the last house.

## Brute force
- Generate all possible subsets of the houses. For each subset, compute the total value. Since there are \(2^n\) subsets for \(n\) houses, this brute-force method is exponential time (\(O(2^n)\)).

## Bottleneck
- The exponential time comes from the naive recursive or brute-force approach. Each house either belongs or doesn’t belong to the subset. This leads to overlapping subproblems, which the DP approach will handle, but it still requires examining every possible combination.

## Key insight
- **Circular constraint**: Because the houses form a circle, the decision about the firs

In [ ]:
import json

# 1. read the 16 held-out test problems
test_rows = []
with open("/content/test.jsonl", encoding="utf-8") as f:
    for line in f:
        test_rows.append(json.loads(line))
print(len(test_rows), "test problems")

# 2. helper: generate ONE derivation for a given problem string
def generate_derivation(problem_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,                 # we need token tensors for generate()
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,              # gives input_ids + attention_mask
    ).to("cuda")

    output = model.generate(
        **inputs,                      # no streamer — we capture the text instead
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        use_cache=True,
    )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


16 test problems


In [ ]:
finetuned_results = []
for i, row in enumerate(test_rows):
    print(i, row["title"])                       # watch progress
    out = generate_derivation(row["problem"])
    finetuned_results.append({
        "title": row["title"],
        "problem": row["problem"],
        "finetuned_output": out,
    })

# 4. save to Drive immediately (crash-safe!)
save_path = "/content/drive/MyDrive/dsa-coach/eval_finetuned.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, ensure_ascii=False, indent=2)

print("done:", len(finetuned_results), "saved to", save_path)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0 Combination Sum


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1 Happy Number


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

13 Merge Two Sorted Lists


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


14 House Robber II


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


15 Search a 2D Matrix
done: 16 saved to /content/drive/MyDrive/dsa-coach/eval_finetuned.json


In [ ]:
import json

# helper: generate ONE derivation using the BASE model (adapter disabled)
def generate_derivation_base(problem_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with model.disable_adapter():          # <-- run as the raw base model
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# loop over the SAME 16 problems (reuse test_rows from Step 1)
base_results = []
for i, row in enumerate(test_rows):
    print(i, row["title"])
    out = generate_derivation_base(row["problem"])
    base_results.append({
        "title": row["title"],
        "problem": row["problem"],
        "base_output": out,
    })

# save to a DIFFERENT file (crash-safe)
save_path = "/content/drive/MyDrive/dsa-coach/eval_base.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(base_results, f, ensure_ascii=False, indent=2)

print("done:", len(base_results), "saved to", save_path)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0 Combination Sum


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1 Happy Number


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

13 Merge Two Sorted Lists


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


14 House Robber II


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


15 Search a 2D Matrix
done: 16 saved to /content/drive/MyDrive/dsa-coach/eval_base.json


In [ ]:
import json

def generate_derivation(problem_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,        # greedy / deterministic
        use_cache=True,
    )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

finetuned_results = []
for i, row in enumerate(test_rows):
    print(i, row["title"])
    finetuned_results.append({
        "title": row["title"], "problem": row["problem"],
        "finetuned_output": generate_derivation(row["problem"]),
    })

with open("/content/drive/MyDrive/dsa-coach/eval_finetuned.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, ensure_ascii=False, indent=2)
print("done finetuned:", len(finetuned_results))

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0 Combination Sum


KeyboardInterrupt: 

In [ ]:
def generate_derivation_base(problem_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")

    with model.disable_adapter():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,        # greedy / deterministic
            use_cache=True,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

base_results = []
for i, row in enumerate(test_rows):
    print(i, row["title"])
    base_results.append({
        "title": row["title"], "problem": row["problem"],
        "base_output": generate_derivation_base(row["problem"]),
    })

with open("/content/drive/MyDrive/dsa-coach/eval_base.json", "w", encoding="utf-8") as f:
    json.dump(base_results, f, ensure_ascii=False, indent=2)
print("done base:", len(base_results))

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0 Combination Sum


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1 Happy Number


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

13 Merge Two Sorted Lists


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


14 House Robber II


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


15 Search a 2D Matrix
done base: 16
